# 05_wbf_infer_submission

원본 Colab 셀에서 분리. (`#@title 74클래스 5-fold WBF 앙상블 추론 → 캐글 제출 CSV 생성`)


In [ ]:
#@title 74클래스 5-fold WBF 앙상블 추론 → 캐글 제출 CSV 생성
#@markdown fold0~4 best 5개 로드 → test 842장 fold별 추론 → WBF 병합 → label_map_74.json으로 K코드 역매핑 → submission CSV 저장. 폴더명 자모결합(NFC/NFD) 차이 자동 흡수

# ============================================================
# [1] 설치
# ============================================================
!pip install -q "rfdetr[train,loggers]" ensemble-boxes        # RF-DETR 추론 + WBF 앙상블 도구

# ============================================================
# [2] Drive 마운트 + import
# ============================================================
from google.colab import drive                                 # 코랩-드라이브 연결 도구
drive.mount("/content/drive", force_remount=True)              # 드라이브 마운트

import os, re, gc, json                                        # 환경변수·정규식·메모리·json 도구
import unicodedata                                             # 유니코드 정규화 도구 (NFC/NFD 차이 흡수)
import numpy as np                                              # 배열 계산 도구
import pandas as pd                                             # 제출 CSV 저장 도구
import torch                                                    # CUDA 메모리 정리 도구
from PIL import Image                                           # 이미지 열기 도구
from pathlib import Path                                        # 경로 객체 도구
from ensemble_boxes import weighted_boxes_fusion                # WBF 앙상블 함수
from rfdetr import RFDETRMedium                                  # 모델 클래스

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # CUDA 메모리 단편화 완화

# ============================================================
# [3] 경로 자동 탐색 (폴더명 자모결합 차이 방지)
# ============================================================
TEAM_ROOT = Path("/content/drive/MyDrive/1팀 공유 문서")         # 팀 공유 루트
USER_ROOT = TEAM_ROOT / "김태윤"                                 # 개인 작업 폴더
TEST_DIR = TEAM_ROOT / "ai12-level1-project" / "sprint_ai_project1_data" / "test_images"  # test 842장
LABEL_MAP_PATH = USER_ROOT / "label_map_74.json"                 # 내부 id -> K코드 역매핑표
OUT_DIR = USER_ROOT / "submission_74cls"                         # 제출 파일 저장 폴더
OUT_DIR.mkdir(parents=True, exist_ok=True)                       # 폴더 생성

def find_dir(parent, keyword):
    # parent 아래에서 keyword 포함 폴더 탐색. 폴더명·키워드 둘 다 NFC 정규화 후 비교(자모결합 차이 흡수)
    key = unicodedata.normalize("NFC", keyword)
    for name in os.listdir(parent):
        if key in unicodedata.normalize("NFC", name) and (parent / name).is_dir():
            return parent / name
    return None

outputs_dir = find_dir(USER_ROOT, "outputs_74")                 # "outputs_74..." 폴더 자동 탐색
assert outputs_dir is not None, f"outputs_74 폴더 못 찾음 (USER_ROOT: {USER_ROOT})"
MODEL_DIR = find_dir(outputs_dir, "베스트모델")                   # 그 안 "베스트모델..." 폴더 탐색
assert MODEL_DIR is not None, f"베스트모델 폴더 못 찾음 (outputs_dir: {outputs_dir})"
print("MODEL_DIR 확정:", MODEL_DIR)                              # 실제 잡힌 경로 확인

NUM_CLASSES = 74                                                 # 74클래스 모델
RESOLUTION = 576                                                 # 학습과 동일 해상도
PRED_THRESHOLD = 0.001                                           # 추론 threshold (후보 넓게 수집)
WBF_IOU_THR = 0.55                                               # WBF 같은 객체 판정 IoU
WBF_SKIP_THR = 0.001                                             # WBF 최저 score 컷
MAX_DETS_PER_IMAGE = 100                                         # 이미지당 최종 예측 상한

assert TEST_DIR.exists(), f"test 폴더 없음: {TEST_DIR}"          # 경로 방어
assert LABEL_MAP_PATH.exists(), f"label_map 없음: {LABEL_MAP_PATH}"

# ============================================================
# [4] 체크포인트 5개 수집 (checkpoint_best_total_XX_74.pth 패턴)
# ============================================================
ckpts = {}                                                       # {fold번호: 체크포인트 경로}
for p in MODEL_DIR.glob("checkpoint_best_total_*_74.pth"):       # 74종 best만 매칭
    m = re.search(r"total_(\d\d)_74", p.name)                    # 'total_00_74' -> '00' 추출
    if m:
        ckpts[int(m.group(1))] = p                               # '00'->0, '01'->1 ... fold 번호

print("발견된 체크포인트:", {k: v.name for k, v in sorted(ckpts.items())})
assert len(ckpts) == 5, f"fold best가 5개가 아님: {len(ckpts)}개"  # 5개 전부 있어야 진행

# ============================================================
# [5] 역매핑표 로드 (내부 id 1~74 -> K코드)
# ============================================================
with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_map = json.load(f)                                     # {str(내부id): K코드}
id2kcode = {int(k): int(v) for k, v in label_map.items()}        # int 키/값으로 변환
assert len(id2kcode) == 74, f"label_map 항목 수 이상: {len(id2kcode)}"
print("역매핑 샘플:", dict(list(id2kcode.items())[:3]))

# ============================================================
# [6] test 이미지 목록
# ============================================================
test_files = sorted(TEST_DIR.glob("*.png"), key=lambda p: int(p.stem))  # 숫자 파일명 순 정렬
print(f"test 이미지 수: {len(test_files)}")                       # 842 확인

# ============================================================
# [7] fold별 추론 (fold 5개 x 842장)
# ============================================================
preds = {fi: {} for fi in ckpts}                                  # preds[fold][image_id] = (boxes_norm, scores, labels)

for fi, ckpt in sorted(ckpts.items()):                             # fold 하나씩
    print(f"\n[fold {fi}] 모델 로드: {ckpt.name}")
    model = RFDETRMedium(                                          # 생성자에 직접 전달 (silent 로딩 버그 방지)
        pretrain_weights=str(ckpt),                                # fine-tuned best 가중치
        num_classes=NUM_CLASSES,                                   # 74 명시
        resolution=RESOLUTION,                                     # 576
    )

    for idx, fp in enumerate(test_files):                          # test 842장 순회
        img = Image.open(fp).convert("RGB")                        # 이미지 로드
        W, H = img.size                                            # 정규화용 원본 크기
        det = model.predict(img, threshold=PRED_THRESHOLD)         # 추론 (supervision Detections)

        if len(det) == 0:                                          # 검출 0이면 빈 배열 저장
            preds[fi][int(fp.stem)] = (np.zeros((0, 4)), np.zeros(0), np.zeros(0, dtype=int))
            continue

        boxes = det.xyxy.copy().astype(float)                      # [x1,y1,x2,y2] 픽셀 좌표
        boxes[:, [0, 2]] /= W                                      # x 좌표 0~1 정규화 (WBF 요구)
        boxes[:, [1, 3]] /= H                                      # y 좌표 0~1 정규화
        boxes = boxes.clip(0, 1)                                   # 경계 밖 값 방어

        preds[fi][int(fp.stem)] = (boxes, det.confidence.copy(), det.class_id.copy())

        if (idx + 1) % 200 == 0:                                   # 진행 로그
            print(f"  {idx + 1}/{len(test_files)}")

    del model; gc.collect(); torch.cuda.empty_cache()              # 다음 fold 위해 GPU 정리
    print(f"[fold {fi}] 추론 완료")

# ============================================================
# [8] WBF 병합 (fold 5벌 -> 1벌) + K코드 역매핑 + CSV 행 생성
# ============================================================
rows = []                                                          # 제출 행 누적
ann_id = 1                                                         # annotation_id 일련번호

for fp in test_files:                                              # 이미지 하나씩 병합
    image_id = int(fp.stem)
    W, H = Image.open(fp).size                                     # 픽셀 복원용 크기

    boxes_list, scores_list, labels_list = [], [], []              # WBF 입력: fold별 리스트
    for fi in sorted(preds):
        b, s, l = preds[fi][image_id]
        boxes_list.append(b.tolist()); scores_list.append(s.tolist()); labels_list.append(l.tolist())

    boxes, scores, labels = weighted_boxes_fusion(                 # 5벌 병합 (fold 가중치 균등)
        boxes_list, scores_list, labels_list,
        iou_thr=WBF_IOU_THR, skip_box_thr=WBF_SKIP_THR,
    )

    order = np.argsort(-scores)[:MAX_DETS_PER_IMAGE]               # score 내림차순 상위 100개
    for i in order:
        x1, y1, x2, y2 = boxes[i]                                  # 정규화 좌표
        bx, by = x1 * W, y1 * H                                     # 픽셀 좌표 복원
        bw, bh = (x2 - x1) * W, (y2 - y1) * H                       # xyxy -> xywh 변환
        kcode = id2kcode[int(labels[i])]                            # 내부 id -> K코드 (대회 category_id)

        rows.append({
            "annotation_id": ann_id, "image_id": image_id,
            "category_id": kcode,
            "bbox_x": round(bx, 2), "bbox_y": round(by, 2),
            "bbox_w": round(bw, 2), "bbox_h": round(bh, 2),
            "score": round(float(scores[i]), 6),
        })
        ann_id += 1

# ============================================================
# [9] CSV 저장
# ============================================================
sub = pd.DataFrame(rows, columns=["annotation_id", "image_id", "category_id",
                                  "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"])
out_path = OUT_DIR / "submission_74cls_wbf.csv"
sub.to_csv(out_path, index=False)                                  # 제출 파일 저장

print(f"\n저장 완료: {out_path}")
print(f"총 행 수: {len(sub)} / 이미지 수: {sub['image_id'].nunique()}")
print(f"category_id 종류: {sub['category_id'].nunique()}")
print(sub.head())